<a href="https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliraza-chaudhary/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Freshness Multiplier

The paper reports that pages updated 31–90 days ago had a 5.43:1 growth-to-decline ratio, and it also reports a separate comparison where refreshed 365+ day pages had higher health and impressions than older stale pages.

My methodology question is: **where does the outcome label come from, and does the comparison support a causal claim?**

The outcome is based on observed growth/decline and measured search-performance metrics, rather than a randomized treatment label. The paper does disclose important windows and cohort definitions, and it describes the result as a pattern study rather than proof of cause and effect. I would therefore ask whether refreshed pages differed from stale pages in other important ways before the refresh, such as prior visibility, content quality, or search demand. The finding is useful as an observed directional signal, but the validation design should not be interpreted as proving that freshness alone caused the improvement.

### Finding 2 — The CTR Cliff

The paper reports that weighted CTR falls sharply as pages move from the top search positions toward deeper position tiers. It reports weighted CTR of 0.420% for the top 3, 0.340% for positions 4–10, 0.325% for positions 11–20, and 0.050% for positions 50+.

My methodology question is: **does the validation design support the broader claim that improving ranking position will cause more clicks?**

The label here is observed CTR grouped by position tier, not an experimental treatment/outcome label. Position and CTR are measured together in search data, so the result shows a strong observed relationship, but it does not by itself establish that moving a page to a higher position would cause the same CTR increase. I would keep the claim as measured/observed click capture by position tier and avoid treating it as a causal estimate of ranking improvement.

In [58]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Before vs after validation design

For the before comparison, I used a random row-level split. This allows pages from the same client to appear in both training and test data, so the estimate may be optimistic when client-specific patterns are shared across the split.

For the after comparison, I used a client-grouped split. The grouped split held out 7 clients from training, with 25 clients used for training. This evaluates the model on clients it did not see during training.

The measured results were higher under the random split: average precision was 0.9391 and ROC-AUC was 0.9207. Under the client-grouped split, average precision was 0.8759 and ROC-AUC was 0.8584. Precision@50 was 1.000 under both splits.

The lower average precision and ROC-AUC under the grouped split indicate that the random-split estimate was more optimistic for these metrics. I therefore treat the client-grouped results as the more useful directional estimate of how the model's ranking performance transfers to unseen clients. These results provide decision-support evidence, not proof that the model will generalize to every future client.

In [59]:
%cd /content/flyrank-ml-internship

!python scripts/01_prepare_features.py

/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [60]:
import os

for root, dirs, files in os.walk("/content"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
/content/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/flyrank-ml-internship/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv
/content/flyrank-ml-internship/flyrank-ml-internship/outputs/refresh_queue_sample.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/mnist_test.csv
/content/sample_data/california_housing_train.csv


In [61]:
!git clone https://github.com/aliraza-chaudhary/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [62]:
!ls -la /content/flyrank-ml-internship

total 108
drwxr-xr-x 13 root root  4096 Aug 26 21:55 .
drwxr-xr-x  1 root root  4096 Aug 26 21:52 ..
-rw-r--r--  1 root root   654 Aug 26 21:52 AGENTS.md
-rw-r--r--  1 root root   654 Aug 26 21:52 CLAUDE.md
drwxr-xr-x  4 root root  4096 Aug 26 21:55 data
-rw-r--r--  1 root root  2763 Aug 26 21:52 DATA_USE.md
drwxr-xr-x  2 root root  4096 Aug 26 21:52 docs
drwxr-xr-x 12 root root  4096 Aug 26 21:55 flyrank-ml-internship
drwxr-xr-x  8 root root  4096 Aug 26 21:52 .git
drwxr-xr-x  3 root root  4096 Aug 26 21:52 .github
-rw-r--r--  1 root root   993 Aug 26 21:52 .gitignore
-rw-r--r--  1 root root 10408 Aug 26 21:52 GUIDE.md
-rw-r--r--  1 root root  1289 Aug 26 21:52 LICENSE
drwxr-xr-x  2 root root  4096 Aug 26 21:52 notebooks
drwxr-xr-x  3 root root  4096 Aug 26 21:52 outputs
-rw-r--r--  1 root root  9860 Aug 26 21:52 README.md
-rw-r--r--  1 root root   107 Aug 26 21:52 requirements.txt
drwxr-xr-x  3 root root  4096 Aug 26 21:55 scripts
-rw-r--r--  1 root root  5671 Aug 26 21:52 SETUP.md
d

In [63]:
!find /content/flyrank-ml-internship/data -type f | sort

/content/flyrank-ml-internship/data/processed/feature_metadata.json
/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv
/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [64]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (30000, 52)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']


In [65]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

TARGET_COL = "is_declining_label"
GROUP_COL = "client_id"

exclude_cols = [
    "content_id",
    "client_id",
    "is_declining_label",
    "trend_direction",
    "trend_pct",
]

FEATURE_COLS = [
    col
    for col in df.select_dtypes(include=np.number).columns
    if col not in exclude_cols
]

X = (
    df[FEATURE_COLS]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df[TARGET_COL].astype(int)


def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean())


def train_and_evaluate(X_train, X_test, y_train, y_test, name):

    model = Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ])

    model.fit(X_train, y_train)

    scores = model.predict_proba(X_test)[:, 1]

    return {
        "validation": name,
        "precision_at_50": precision_at_k(
            y_test, scores, 50
        ),
        "average_precision": average_precision_score(
            y_test, scores
        ),
        "roc_auc": roc_auc_score(
            y_test, scores
        ),
        "test_rows": len(y_test)
    }


# BEFORE: random row split
X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

before = train_and_evaluate(
    X_train_random,
    X_test_random,
    y_train_random,
    y_test_random,
    "Before: random row split"
)


# AFTER: client-grouped split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y,
        groups=df[GROUP_COL]
    )
)

X_train_grouped = X.iloc[train_idx]
X_test_grouped = X.iloc[test_idx]

y_train_grouped = y.iloc[train_idx]
y_test_grouped = y.iloc[test_idx]

after = train_and_evaluate(
    X_train_grouped,
    X_test_grouped,
    y_train_grouped,
    y_test_grouped,
    "After: client-grouped split"
)


comparison = pd.DataFrame([
    before,
    after
])

display(comparison)


# Sanity check
train_clients = set(df.iloc[train_idx][GROUP_COL])
test_clients = set(df.iloc[test_idx][GROUP_COL])

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))

assert train_clients.isdisjoint(test_clients)

print("Client-holdout sanity check passed.")

,validation,precision_at_50,average_precision,roc_auc,test_rows
0,Before: random row split,1.0,0.939074,0.920684,6000
1,After: client-grouped split,1.0,0.875856,0.858406,6163


Train clients: 25
Test clients: 7
Client-holdout sanity check passed.


In [66]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage audit

I checked the final feature set for direct target leakage, trend-derived fields, and identifier columns. is_declining_label is excluded because it is the prediction target. trend_direction and trend_pct are excluded because they describe the outcome used to construct the label. content_id and client_id are excluded because they are identifiers rather than model inputs.

I also checked feature names for terms associated with labels, targets, decline, and trends. This is a screening check rather than proof that every possible form of leakage is absent, so the conclusion is limited to the checks performed here.

In [67]:
# ============================================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================================

print("Target column:", TARGET_COL)
print("Number of model features:", len(FEATURE_COLS))

# ------------------------------------------------------------
# 1. Direct target leakage
# ------------------------------------------------------------

direct_leakage = [
    col for col in FEATURE_COLS
    if col in [
        "is_declining_label",
        "trend_direction",
        "trend_pct"
    ]
]

print("\nDirect target/trend leakage:")
print(direct_leakage)

assert len(direct_leakage) == 0


# ------------------------------------------------------------
# 2. Identifier leakage
# ------------------------------------------------------------

identifier_features = [
    col for col in FEATURE_COLS
    if col in [
        "content_id",
        "client_id"
    ]
]

print("\nIdentifier columns included as features:")
print(identifier_features)

assert len(identifier_features) == 0


# ------------------------------------------------------------
# 3. Suspicious feature names
# ------------------------------------------------------------

suspicious_terms = [
    "label",
    "target",
    "declin",
    "trend"
]

suspicious_features = [
    col
    for col in FEATURE_COLS
    if any(term in col.lower() for term in suspicious_terms)
]

print("\nPotentially suspicious feature names:")
print(suspicious_features)


# ------------------------------------------------------------
# 4. Final audit
# ------------------------------------------------------------

print("\nFinal feature list:")
print(FEATURE_COLS)

print("\nLeakage audit completed.")
print(
    "Target, trend-derived, and identifier columns "
    "are excluded from the model feature set."
)

Target column: is_declining_label
Number of model features: 36

Direct target/trend leakage:
[]

Identifier columns included as features:
[]

Potentially suspicious feature names:
[]

Final feature list:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'has_clicks', 'has_ai_sessions', 'measurable_opportunity']

Leakage audit completed.
Target, trend-derived, and identifier columns are excluded from the model feature set.


In [68]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

4. Claim rewrite
Original claim

The model can identify pages that will decline and reliably prioritize the best pages for refresh.

Safer claim

The model produced a measured ranking of pages associated with observed decline. Under the client-grouped validation, it achieved 0.8759 average precision and 0.8584 ROC-AUC, providing directional evidence about ranking performance on unseen clients. The output should be treated as decision-support for refresh prioritization rather than proof that the model will predict future traffic declines or that refreshing a selected page will cause improvement.

In [69]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.